In [1]:
import pandas as pd
import numpy as np
from src.model_pipeline import ModelPipeline

In [2]:
df_final = pd.read_parquet("data/ready_to_train/CREDIT_df_final_80_feats_20250627.parquet")

In [3]:
categorical_cols = [
    'CustomerSex',
    'MCC Category',
    # 'POSMode',
    # 'CardProductGrouped',
    # 'CurrencyCodeGroup'
]
numerical_cols = [
    col for col in df_final.columns if col not in (categorical_cols + ['Confirmed','Transaction Datetime'])
]

In [4]:
top_numerical_cols = [
    'AvgDurationSinceFirstTrnxToCurrentMCCL30D',
    'TxnCountToPOSModeL15min',
    'RatioTxnCountToPOSModeL30DL15min',
    'CntUnique_CardNo_by_currencyCode_L30D',
    'RatioTxnCountL30DL15min',
    'RatioCntUnique_CardNo_by_currencyCodeL30DL15min',
    'Transaction Amount', # Channel
    'CntUnique_MCC_by_CardNo_L15M',
    'time_diff',
    'RatioCntUnique_MCC_by_CardNoL30DL15min',
    'TxnCount_L15M',
    'RatioTxnCount_to_countryCodeL30DL15min',
    'RatioTxnCount_to_MCCL30DL15min',
    'RatioSum_Amt_to_MCCL30DL15min',
    'TxnCount_to_countryCode_L15M',
    'Max_Amt_to_MCC_L30D',
    'AvgDurationSinceFirstTrnxToCurrentMCCL15min',
    'TotalTrxAmount10Mi', # TSCF
    'IsTop10HighRiskMCCLast30D',
    # 'RatioAvgDurationSinceFirstTrnxToCurrentMCCL30DL15min',
    # 'RatioSum_Amt_to_countryCodeL30DL15min',
    # 'Sum_Amt_to_MCC_L30D',
    # 'AvgDurationSinceFirstTrnxToCurrentCountryCodeL30D',
    # 'TotalTrxAmount15Mi',
    # 'CntUnique_CardNo_by_MCC_L30D'
]

In [22]:
len(top_numerical_cols + categorical_cols)

21

In [23]:
set(top_numerical_cols + categorical_cols)

{'AvgDurationSinceFirstTrnxToCurrentMCCL15min',
 'AvgDurationSinceFirstTrnxToCurrentMCCL30D',
 'CntUnique_CardNo_by_currencyCode_L30D',
 'CntUnique_MCC_by_CardNo_L15M',
 'CustomerSex',
 'IsTop10HighRiskMCCLast30D',
 'MCC Category',
 'Max_Amt_to_MCC_L30D',
 'RatioCntUnique_CardNo_by_currencyCodeL30DL15min',
 'RatioCntUnique_MCC_by_CardNoL30DL15min',
 'RatioSum_Amt_to_MCCL30DL15min',
 'RatioTxnCountL30DL15min',
 'RatioTxnCountToPOSModeL30DL15min',
 'RatioTxnCount_to_MCCL30DL15min',
 'RatioTxnCount_to_countryCodeL30DL15min',
 'TotalTrxAmount10Mi',
 'Transaction Amount',
 'TxnCountToPOSModeL15min',
 'TxnCount_L15M',
 'TxnCount_to_countryCode_L15M',
 'time_diff'}

In [6]:
df_final = df_final[['Confirmed','Transaction Datetime'] + top_numerical_cols + categorical_cols]

# Rename Column

In [20]:
renamed_column_map = {
    'AvgDurationSinceFirstTrnxToCurrentMCCL15min': 'AvgTimeFirstTxnToCurrentMCCL15min',
    'AvgDurationSinceFirstTrnxToCurrentMCCL30D': 'AvgTimeFirstTxnToCurrentMCCL30D',
    'CntUnique_CardNo_by_currencyCode_L30D': 'CntUniqueCardNoByCurrencyCodeL30D',
    'CntUnique_MCC_by_CardNo_L15M': 'CntUniqueMCCByCardNoL15min',
    'MCC Category': 'MCCCategory',
    'Max_Amt_to_MCC_L30D': 'MaxAmtToMCCL30D',
    'RatioAvgDurationSinceFirstTrnxToCurrentMCCL30DL15min': 'RatioAvgTimeFirstTxnToCurrentMCCL30DL15min',
    'RatioCntUnique_CardNo_by_currencyCodeL30DL15min': 'RatioCntUniqueCardNoByCurrencyCodeL30DL15min',
    'RatioCntUnique_MCC_by_CardNoL30DL15min': 'RatioCntUniqueMCCByCardNoL30DL15min',
    'RatioSum_Amt_to_MCCL30DL15min': 'RatioSumAmtToMCCL30DL15min',
    'RatioTxnCount_to_MCCL30DL15min': 'RatioTxnCountToMCCL30DL15min',
    'RatioTxnCount_to_countryCodeL30DL15min': 'RatioTxnCountToCountryCodeL30DL15min',
    'Transaction Amount': 'TransactionAmount',
    'TxnCount_L15M': 'TxnCountL15min',
    'TxnCount_to_countryCode_L15M': 'TxnCountToCountryCodeL15min',
    'time_diff': 'TxnTimeDifference',
}
df_final = df_final.rename(columns=renamed_column_map)

# Model Train

In [29]:
df_final.to_parquet("data/ready_to_train/df_credit_final_21_feats_20250702.parquet")

In [30]:
# setup pipeline
pipeline = ModelPipeline(
    model_type="random_forest",
    random_state=42
)

split_date = "2025-05-01"
df_splits = pipeline.split_data_by_date(
    df=df_final,
    date_column="Transaction Datetime",
    split_date=split_date
)

Initializing ModelPipeline with model type: random_forest
Retrieving model instance for type: random_forest
Splitting data by date (pre-preprocessing)...
Train samples: 1197282, Test samples: 201546
Data splitting complete.


In [31]:
# prepare TRAIN data
target_col = "Confirmed"
X, y = pipeline.prepare_data(
    df=df_splits["df_train"],
    target_column=target_col,
    exclude_columns=["Transaction Datetime"],
    is_apply_one_hot=True,
    is_apply_log=False,
    is_impute_median=False,
    is_training=True
)

Preparing data (is_training=True)...
Starting core numeric and boolean preprocessing...
Core numeric and boolean preprocessing complete.
Fitting CategoryManager...
  Fitted categories for column 'CustomerSex': ['0', '1', '2', '__missing__']
  Fitted categories for column 'MCCCategory': ['AIRLINES, AIR CARRIERS', 'ASSOCIATIONS/ORGANIZATIONS', 'BUSINESS/PROFESSIONAL/MISCELLANEOUS SERVICES', 'CAR RENTAL AGENCIES', 'CLOTHING/SHOES/ACCESSORIES/UNIFORMS/RETAIL STORES/BUYING AND SELLING SERVICES', 'EDUCATIONAL SERVICES', 'ELECTRONIC AND TECHNICAL SERVICES', 'ENTERTAINMENT/THEATER/DANCE STUDIOS', 'FINANCIAL SERVICES', 'FURNISHINGS/APPLIANCES/MAINTENANCE/HOME', 'GROCERY STORES/PHARMACIES/FOOD SERVICES/RESTAURANTS', 'HEALTHCARE/CHILD SERVICES', 'LODGING — HOTELS, MOTELS, RESORTS', 'OTHER', 'PERSONAL SERVICES', 'TRAVEL/TRANSPORTATION/GAS AND FUEL SERVICES', '__missing__']
CategoryManager fitting complete.
Transforming data using CategoryManager...
CategoryManager transformation complete.
Applying

In [32]:
# split data
X_train, X_test, y_train, y_test = pipeline.split_data(X, y, test_size=0.3)

Splitting data into train and test sets...
Data splitting complete.


## Randomized

In [33]:
from src.imbalance_learn import ImbalancedSampler

# OverSampling
ros_sampler = ImbalancedSampler(
    algorithm='RandomOverSampler',
    sampler_type='upsample',
    target_ratio=0.2, # 7%
    random_state=1234
)

X_train_ros, y_train_ros = ros_sampler.fit_resample(X_train, y_train)

Original dataset shape: Counter({False: 831875, True: 6222})
Resampled dataset shape using RandomOverSampler: Counter({False: 831875, True: 166375})


## SMOTE

In [19]:
# OverSampling
smote_sampler = ImbalancedSampler(
    algorithm='SMOTE',
    sampler_type='upsample',
    target_ratio=0.2, # 7%
    random_state=1234
)

X_train_smote, y_train_smote = smote_sampler.fit_resample(X_train, y_train)

Original dataset shape: Counter({False: 831875, True: 6222})
Resampled dataset shape using SMOTE: Counter({False: 831875, True: 166375})


## Train ROS

In [34]:
# build and train
pipeline.build_pipeline()
pipeline.train(X_train_ros, y_train_ros, show_training_log=False)

Building pipeline...
Pipeline built.
Training model...
Model training complete.


In [35]:
base_rf_ros = pipeline.get_model()

Returning trained model...


In [36]:
import pickle

filepath = "model/credit/base_rf_ros_38_feats_20250702.pkl"
with open(filepath, "wb") as f:
    pickle.dump(base_rf_ros, f)

In [37]:
# Evaluate
test_results = pipeline.evaluate(X_test, y_test)

Evaluating model...
Accuracy: 0.9742
AUC: 0.9902
PR AUC: 0.6981
Precision: 0.2106
Recall: 0.9003

Classification Report:
               precision    recall  f1-score   support

       False       1.00      0.97      0.99    356518
        True       0.21      0.90      0.34      2667

    accuracy                           0.97    359185
   macro avg       0.60      0.94      0.66    359185
weighted avg       0.99      0.97      0.98    359185



## Train SMOTE

In [20]:
# build and train
pipeline.build_pipeline()
pipeline.train(X_train_smote, y_train_smote, show_training_log=False)

Building pipeline...
Pipeline built.
Training model...
Model training complete.


In [21]:
base_rf_smote = pipeline.get_model()

Returning trained model...


In [22]:
# Evaluate
test_results_smote = pipeline.evaluate(X_test, y_test)

Evaluating model...
Accuracy: 0.9780
AUC: 0.9883
PR AUC: 0.6365
Precision: 0.2336
Recall: 0.8628

Classification Report:
               precision    recall  f1-score   support

       False       1.00      0.98      0.99    356518
        True       0.23      0.86      0.37      2667

    accuracy                           0.98    359185
   macro avg       0.62      0.92      0.68    359185
weighted avg       0.99      0.98      0.98    359185



# BackTesting

In [16]:
df_credit_mix = pd.read_parquet("data/base/df_credit_clean_dec24_may25.parquet")
trx_id_list = list(df_credit_mix[df_credit_mix.Confirmed.isin([0,1])]['Transaction Serial No'])
df_oos_new = df_splits["df_test"][df_splits["df_test"].index.isin(trx_id_list)]

In [17]:
X_oos, y_oos = pipeline.prepare_data(
    df=df_splits["df_test"],
    # df=df_oos_new,
    target_column=target_col,
    exclude_columns=["Transaction Datetime"],
    is_apply_one_hot=True,
    is_apply_log=False,
    is_impute_median=False,
    is_training=False
)

Preparing data (is_training=False)...
Starting core numeric and boolean preprocessing...
Core numeric and boolean preprocessing complete.
Transforming data using CategoryManager...
CategoryManager transformation complete.
Applying One-Hot Encoders for column: Index(['CustomerSex', 'MCCCategory'], dtype='object')...
Identified numerical columns for imputation: ['AvgDurationSinceFirstTrnxToCurrentMCCL30D', 'TxnCountToPOSModeL15min', 'RatioTxnCountToPOSModeL30DL15min', 'CntUniqueCardNoByCurrencyCodeL30D', 'RatioTxnCountL30DL15min', 'RatioCntUniqueCardNoByCurrencyCodeL30DL15min', 'TransactionAmount', 'CntUniqueMCCByCardNoL15min', 'TxnTimeDifference', 'RatioCntUniqueMCCByCardNoL30DL15min', 'TxnCountL15min', 'RatioTxnCountToCountryCodeL30DL15min', 'RatioTxnCountToMCCL30DL15min', 'RatioSumAmtToMCCL30DL15min', 'TxnCountToCountryCodeL15min', 'MaxAmtToMCCL30D', 'AvgDurationSinceFirstTrnxToCurrentMCCL15min', 'TotalTrxAmount10Mi', 'IsTop10HighRiskMCCLast30D']
Identified categorical columns (encode

In [18]:
# Evaluate ALL trx (base RF ROS)
oos_results = pipeline.evaluate(X_oos, y_oos)

Evaluating model...
Accuracy: 0.9838
AUC: 0.9815
PR AUC: 0.5768
Precision: 0.2802
Recall: 0.7580

Classification Report:
               precision    recall  f1-score   support

       False       1.00      0.99      0.99    200054
        True       0.28      0.76      0.41      1492

    accuracy                           0.98    201546
   macro avg       0.64      0.87      0.70    201546
weighted avg       0.99      0.98      0.99    201546



In [19]:
len(X.columns)

38

## SMOTE Backtest

In [23]:
# SMOTE RF
oos_results_smote = pipeline.evaluate(X_oos, y_oos)

Evaluating model...
Accuracy: 0.9851
AUC: 0.9816
PR AUC: 0.5169
Precision: 0.2911
Recall: 0.7078

Classification Report:
               precision    recall  f1-score   support

       False       1.00      0.99      0.99    200054
        True       0.29      0.71      0.41      1492

    accuracy                           0.99    201546
   macro avg       0.64      0.85      0.70    201546
weighted avg       0.99      0.99      0.99    201546



# Feature Importance

In [19]:
len(top_25_numerical_cols + selected_categorical_cols), len(X.columns)

(23, 47)